# SOSD stability charts on the Colab GPU — streamed app

The chart is computed **and coloured** on the Colab GPU by a Julia worker that stays alive between
frames; this notebook streams every refinement level to the page as a JPEG (the pattern of the
InterpolatedNyquist T4 app). The page has the model text, examples and parameters of the WebGPU
page and only draws: axes, colour bar, zoom (mouse wheel) and pan (drag).

* Levels: the chart is refined on nested lattices down to one ρ per pixel of the final grid
  (the plot area, or HD / Full HD / 4K / 8K for export). While you drag a slider or the chart, the
  start level is chosen from the measured GPU rate so that a frame takes ~1/fps.
* Solver: one m-step Arnoldi pass per point on the GPU (the algorithm of the WebGPU page), Float32
  by default; *precision* can also be Float64, or Float32 with the last level in Float64 (on a T4,
  Float64 is ~30× slower; on an A100 only ~2×).
* *forced response*: stable points coloured by the peak-to-peak amplitude on the periodic orbit.

1. *Runtime → Change runtime type → GPU* (T4).
2. Run the cells in order. The first start installs Julia and compiles the GPU kernels (5–10 min
   on a fresh runtime); a new model, s, m or precision compiles once more (10–60 s).
3. When done: *Runtime → Disconnect and delete runtime* (compute units).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
# Julia (juliaup) + the SOSD.jl `gpu` branch + its GPU environment (log: /content/stream_setup.log)
export PATH=/root/.juliaup/bin:$PATH
if ! command -v julia >/dev/null; then
  curl -fsSL https://install.julialang.org | sh -s -- --yes --default-channel 1.12 > /content/juliaup.log 2>&1
fi
if [ -d /content/SOSD.jl/.git ]; then
  cd /content/SOSD.jl && git fetch -q --depth 1 origin gpu && git reset -q --hard FETCH_HEAD
else
  git clone -q -b gpu --depth 1 https://github.com/bachrathyd/SOSD.jl /content/SOSD.jl
fi
cd /content/SOSD.jl && git log -1 --format='code: %h %s'
julia --project=gpu/webui -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()' > /content/stream_setup.log 2>&1
tail -n 3 /content/stream_setup.log
echo "setup done"

In [ ]:
# start the GPU worker (first time: compiles the kernels) and show the app;
# re-running this cell keeps the running worker and restarts only the page
exec(open('/content/SOSD.jl/gpu/stream/app.py').read())

Worker log: `!tail -n 30 /content/sosd_stream_worker.log`. Saved PNGs are in `/content` (Files panel).
If the page stays empty, re-run the last cell.